# 03 — Operational Interpretation and Insights

This notebook translates the network metrics computed in Notebook 02 into
operational insights about hub dominance, network resilience, fleet
requirements and the effect of fuel price on network evolution.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.data_preprocessing import load_data, correct_coordinates
from src.network_construction import build_graph
from src.network_metrics import assortativity, kplus_core

In [ ]:
airports, flights = load_data('../data/Airports.csv', '../data/Flight Data.xlsx')
airports = correct_coordinates(airports)

countries = ['USA', 'China', 'United Kingdom', 'Australia']
graphs = {c: build_graph(flights, c) for c in countries}

## Summary metrics

| Country | Assortativity r | Core size | Structure type |
|---------|----------------:|----------:|----------------|
| USA     | −0.198          | ~41       | Hub–spoke with critical bridge airports |
| China   | −0.397          | ~23       | Strongly hub-focused, highly disassortative |
| UK      | −0.119          | ~16       | Multi-hub / distributed, more resilient |
| Australia | −0.230        | ~9        | Hub–spoke around a few coastal hubs |

In [ ]:
summary = []
for c in countries:
    G = graphs[c]
    r_val, _, _, _, _ = assortativity(G)
    _, _, peak_rank, _ = kplus_core(G)
    summary.append({
        'Country': c,
        'Nodes': len(G.nodes),
        'Edges': len(G.edges),
        'Assortativity (r)': round(r_val, 3),
        'Core size (r*)': peak_rank
    })

pd.DataFrame(summary)

## Key findings

### Hub dominance and transfer roles

- **USA and China** show strong degree–betweenness correlation, confirming
  hub–spoke dominance where top hubs carry most transfer traffic.
- A **critical bridge airport** with moderate degree but highest betweenness
  is identified in the USA, likely serving geographically isolated regions
  (e.g. Alaska/Hawaii connections).
- **UK** shows weaker correlation — a multi-hub system where no single airport
  dominates transfers. Short domestic distances and competing transport modes
  (rail) distribute traffic.

### Network resilience

- All networks are **disassortative** (r < 0): hubs connect to small airports,
  not to each other. This is efficient but creates vulnerability at hub nodes.
- **China** is most vulnerable (r = −0.397): extreme hub-dependence means
  failure of a single major hub has outsized impact.
- **UK** is most resilient (r = −0.119): traffic is distributed, so no single
  failure point dominates.

### Core size and economic scale

Larger economies tend to develop larger network cores. This provides higher
structural robustness (more airports share core functions) at the cost of
lower economic efficiency.

| Country | GDP rank (2025) | Core size |
|---------|----------------:|----------:|
| USA     | 1               | ~41       |
| China   | 2               | ~23       |
| UK      | 6               | ~16       |
| Australia | 15            | ~9        |

## Fuel price effects on network evolution

Using the **Random Geometric Graph** (RGG) framework, where connection
probability between airports depends on geographic distance and a fuel-price
penalty parameter α:

$$Q_{ij} = \frac{d_{ij}^{-\alpha}}{\sum_{k,l} d_{kl}^{-\alpha}}$$

### High fuel price (high α)
- Long-distance routes become uneconomical
- Hub–spoke structure weakens
- Networks shift towards spatial point-to-point structures
- Small/medium fuel-efficient aircraft preferred

### Low fuel price (low α)
- Long-distance hub connections are reinforced
- Degree–betweenness correlation strengthens
- Large-capacity aircraft needed at major hubs

### Country-specific sensitivity
- **USA and Australia** (geographically large): highly sensitive to fuel price
  — long-distance routes are vulnerable to α increases.
- **UK** (compact geography): relatively insensitive — most routes remain
  viable regardless of moderate fuel price changes.
- **China**: eastern hub concentration provides some buffer, but western
  connectivity is sensitive to fuel costs.

### Limits of hub growth
Hub growth is constrained by a truncated power-law distribution in real
networks — physical capacity, slot limitations and economic constraints
prevent any airport from growing indefinitely.